# 05 — End-to-End Data Pipeline

This notebook ties all four services into one realistic pipeline scenario:

> **Scenario:** Ingest raw order data from a CSV upload into S3, catalog it in Glue, query it via Athena, provision an EC2 spot fleet to represent batch workers, and write processed output as Parquet.

Architecture:
```
CSV upload → S3 (raw)  →  Glue Catalog  →  Athena query  →  S3 (processed Parquet)
                                               ↑
                                    EC2 Fleet (batch workers)
```

This is the kind of task you might be asked to whiteboard or code live in a DE interview.

In [1]:
import sys, datetime, io, time
sys.path.insert(0, "..")
import helpers

import boto3
import pandas as pd
from moto import mock_aws

---
## Step 1 — Stand up infrastructure

In [2]:
# All steps run inside one mock_aws() context so state persists across cells.
# We start the mock here and use .start() / .stop() explicitly.
from moto import mock_aws

mock = mock_aws()
mock.start()

# Create all clients
s3     = helpers.make_boto3_client("s3")
glue   = helpers.make_boto3_client("glue")
athena = helpers.make_boto3_client("athena")
ec2    = helpers.make_boto3_client("ec2")

print("Mock started, clients ready.")

Mock started, clients ready.


In [3]:
# Infrastructure: VPC, subnet, security group
vpc_id    = ec2.create_vpc(CidrBlock="10.0.0.0/16")["Vpc"]["VpcId"]
subnet_id = ec2.create_subnet(VpcId=vpc_id, CidrBlock="10.0.1.0/24")["Subnet"]["SubnetId"]
sg_id     = ec2.create_security_group(GroupName="batch-sg", Description="Batch workers", VpcId=vpc_id)["GroupId"]

print(f"VPC: {vpc_id}, Subnet: {subnet_id}, SG: {sg_id}")

VPC: vpc-9024a21a1deacfd24, Subnet: subnet-eb833a001dc31cb74, SG: sg-8bc303c0e742dbd73


---
## Step 2 — Create S3 buckets

In [4]:
RAW_BUCKET       = "pipeline-raw-data"
PROCESSED_BUCKET = "pipeline-processed-data"
RESULTS_BUCKET   = "pipeline-athena-results"

for bucket in [RAW_BUCKET, PROCESSED_BUCKET, RESULTS_BUCKET]:
    s3.create_bucket(Bucket=bucket)
    print(f"Created: s3://{bucket}")

Created: s3://pipeline-raw-data
Created: s3://pipeline-processed-data
Created: s3://pipeline-athena-results


---
## Step 3 — Upload partitioned raw CSV data

In [5]:
# Generate synthetic order data for 3 days
import random
random.seed(42)

statuses = ["completed", "pending", "cancelled"]
all_dates = []

for day_offset in range(3):
    date = datetime.date(2024, 1, 1) + datetime.timedelta(days=day_offset)
    dt_str = str(date)

    # Create daily records
    records = [
        {
            "order_id":    day_offset * 100 + i,
            "customer_id": random.randint(1, 50),
            "amount":      round(random.uniform(10, 500), 2),
            "status":      random.choice(statuses),
        }
        for i in range(10)
    ]
    df = pd.DataFrame(records)

    # Hive-style partition key: year=YYYY/month=MM/day=DD/
    key = f"orders/year={date.year}/month={date.month:02d}/day={date.day:02d}/data.csv"
    helpers.upload_df_as_csv(s3, df, RAW_BUCKET, key)
    all_dates.append(dt_str)
    print(f"Uploaded {len(records)} records → s3://{RAW_BUCKET}/{key}")

print(f"\nTotal partitions: {len(all_dates)}")

Uploaded 10 records → s3://pipeline-raw-data/orders/year=2024/month=01/day=01/data.csv
Uploaded 10 records → s3://pipeline-raw-data/orders/year=2024/month=01/day=02/data.csv
Uploaded 10 records → s3://pipeline-raw-data/orders/year=2024/month=01/day=03/data.csv

Total partitions: 3


---
## Step 4 — Register table in Glue Catalog

In [6]:
glue.create_database(DatabaseInput={"Name": "pipeline_db"})

glue.create_table(
    DatabaseName="pipeline_db",
    TableInput={
        "Name": "orders",
        "StorageDescriptor": {
            "Location": f"s3://{RAW_BUCKET}/orders/",
            "Columns": [
                {"Name": "order_id",    "Type": "bigint"},
                {"Name": "customer_id", "Type": "bigint"},
                {"Name": "amount",      "Type": "double"},
                {"Name": "status",      "Type": "string"},
            ],
            "InputFormat":  "org.apache.hadoop.mapred.TextInputFormat",
            "OutputFormat": "org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat",
            "SerdeInfo": {
                "SerializationLibrary": "org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe",
                "Parameters": {"field.delim": ",", "skip.header.line.count": "1"},
            },
        },
        "PartitionKeys": [
            {"Name": "year",  "Type": "int"},
            {"Name": "month", "Type": "int"},
            {"Name": "day",   "Type": "int"},
        ],
        "TableType": "EXTERNAL_TABLE",
        "Parameters": {"classification": "csv"},
    },
)
print("Glue table 'orders' created in 'pipeline_db'")

Glue table 'orders' created in 'pipeline_db'


---
## Step 5 — Register partitions

In [7]:
import datetime

partition_inputs = []
for day_offset in range(3):
    d = datetime.date(2024, 1, 1) + datetime.timedelta(days=day_offset)
    location = f"s3://{RAW_BUCKET}/orders/year={d.year}/month={d.month:02d}/day={d.day:02d}/"
    partition_inputs.append({
        "Values": [str(d.year), str(d.month), str(d.day)],
        "StorageDescriptor": {
            "Location": location,
            "InputFormat": "org.apache.hadoop.mapred.TextInputFormat",
            "OutputFormat": "org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat",
            "SerdeInfo": {"SerializationLibrary": "org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe"},
            "Columns": [],
        },
    })

glue.batch_create_partition(
    DatabaseName="pipeline_db",
    TableName="orders",
    PartitionInputList=partition_inputs,
)

parts = glue.get_partitions(DatabaseName="pipeline_db", TableName="orders")["Partitions"]
print(f"Registered {len(parts)} partitions:")
for p in parts:
    print(f"  year={p['Values'][0]} month={p['Values'][1]} day={p['Values'][2]}")

Registered 3 partitions:
  year=2024 month=1 day=1
  year=2024 month=1 day=2
  year=2024 month=1 day=3


---
## Step 6 — Set up Athena workgroup and run a query

In [8]:
# Create workgroup
athena.create_work_group(
    Name="pipeline-wg",
    Configuration={
        "ResultConfiguration": {"OutputLocation": f"s3://{RESULTS_BUCKET}/"}
    },
)

# Inject mock results (Athena doesn't execute real SQL in moto)
helpers.inject_athena_results(
    rows=[
        ["completed", "15", "3250.75"],
        ["pending",   "8",  "1100.00"],
        ["cancelled", "7",  "980.50"],
    ],
    column_info=[
        {"Name": "status",      "Type": "varchar"},
        {"Name": "order_count", "Type": "integer"},
        {"Name": "total_amount","Type": "double"},
    ],
)

exec_id = athena.start_query_execution(
    QueryString="SELECT status, COUNT(*) AS order_count, SUM(amount) AS total_amount FROM pipeline_db.orders GROUP BY status",
    QueryExecutionContext={"Database": "pipeline_db"},
    WorkGroup="pipeline-wg",
    ResultConfiguration={"OutputLocation": f"s3://{RESULTS_BUCKET}/"},
)["QueryExecutionId"]

print("Query started:", exec_id)

Query started: ae9abd9a-74df-4f2a-bcb5-0df03e69976f


In [9]:
# Poll and get results
for _ in range(10):
    state = athena.get_query_execution(QueryExecutionId=exec_id)["QueryExecution"]["Status"]["State"]
    if state in {"SUCCEEDED", "FAILED", "CANCELLED"}:
        break
    time.sleep(0.2)

print("Final state:", state)

raw_results = athena.get_query_results(QueryExecutionId=exec_id)
results_df = helpers.parse_athena_results(raw_results)
print("\nQuery results:")
print(results_df)

Final state: SUCCEEDED

Query results:
      status order_count total_amount
0  completed          15      3250.75
1    pending           8      1100.00
2  cancelled           7       980.50


---
## Step 7 — Provision EC2 Fleet for batch workers

In [10]:
# Create the launch template first, then the fleet
ec2.create_launch_template(
    LaunchTemplateName="batch-lt",
    LaunchTemplateData={"ImageId": "ami-12345678", "InstanceType": "m5.xlarge"},
)

fleet_id = ec2.create_fleet(
    LaunchTemplateConfigs=[{
        "LaunchTemplateSpecification": {"LaunchTemplateName": "batch-lt", "Version": "$Latest"},
        "Overrides": [
            {"InstanceType": "m5.xlarge",  "SubnetId": subnet_id},
            {"InstanceType": "m5.2xlarge", "SubnetId": subnet_id},
        ],
    }],
    TargetCapacitySpecification={
        "TotalTargetCapacity": 5,
        "OnDemandTargetCapacity": 2,
        "SpotTargetCapacity": 3,
        "DefaultTargetCapacityType": "spot",
    },
    Type="instant",
    TagSpecifications=[{
        "ResourceType": "fleet",
        "Tags": [{"Key": "Pipeline", "Value": "daily-orders-etl"}],
    }],
)["FleetId"]

fleet_desc = ec2.describe_fleets(FleetIds=[fleet_id])["Fleets"][0]
tcs = fleet_desc["TargetCapacitySpecification"]
print(f"Fleet {fleet_id}: {tcs['TotalTargetCapacity']} total ({tcs['OnDemandTargetCapacity']} on-demand + {tcs['SpotTargetCapacity']} spot)")

Fleet fleet-92337290-c843-f00a-171f-f857759c2c94: 5 total (2 on-demand + 3 spot)


---
## Step 8 — Simulate batch transformation: write processed Parquet

In [11]:
# In a real pipeline the EC2/Spark workers would do this.
# Here we simulate by reading the raw CSVs and writing aggregated Parquet.

all_dfs = []
for day_offset in range(3):
    d = datetime.date(2024, 1, 1) + datetime.timedelta(days=day_offset)
    key = f"orders/year={d.year}/month={d.month:02d}/day={d.day:02d}/data.csv"
    df = helpers.download_df_from_csv(s3, RAW_BUCKET, key)
    df["date"] = str(d)
    all_dfs.append(df)

raw_df = pd.concat(all_dfs, ignore_index=True)
print(f"Read {len(raw_df)} raw records")

# Aggregate: total amount and count per status per day
processed_df = (
    raw_df
    .groupby(["date", "status"])
    .agg(order_count=("order_id", "count"), total_amount=("amount", "sum"))
    .reset_index()
)
print("\nAggregated:")
print(processed_df.to_string(index=False))

# Write as Parquet to processed bucket
helpers.upload_df_as_parquet(s3, processed_df, PROCESSED_BUCKET, "orders_summary/data.parquet")
print("\nWritten to s3://{}/orders_summary/data.parquet".format(PROCESSED_BUCKET))

Read 30 raw records

Aggregated:
      date    status  order_count  total_amount
2024-01-01 cancelled            4        637.05
2024-01-01 completed            5        987.98
2024-01-01   pending            1         52.60
2024-01-02 cancelled            3       1111.26
2024-01-02 completed            3        336.39
2024-01-02   pending            4       1529.67
2024-01-03 cancelled            6       1683.78
2024-01-03 completed            2        852.11
2024-01-03   pending            2        179.77

Written to s3://pipeline-processed-data/orders_summary/data.parquet


---
## Step 9 — Register processed output table in Glue

In [12]:
glue.create_table(
    DatabaseName="pipeline_db",
    TableInput={
        "Name": "orders_summary",
        "StorageDescriptor": {
            "Location": f"s3://{PROCESSED_BUCKET}/orders_summary/",
            "Columns": [
                {"Name": "date",         "Type": "string"},
                {"Name": "status",       "Type": "string"},
                {"Name": "order_count",  "Type": "bigint"},
                {"Name": "total_amount", "Type": "double"},
            ],
            "InputFormat":  "org.apache.hadoop.hive.ql.io.parquet.MapredParquetInputFormat",
            "OutputFormat": "org.apache.hadoop.hive.ql.io.parquet.MapredParquetOutputFormat",
            "SerdeInfo": {
                "SerializationLibrary": "org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe",
                "Parameters": {"serialization.format": "1"},
            },
        },
        "PartitionKeys": [],
        "TableType": "EXTERNAL_TABLE",
        "Parameters": {"classification": "parquet"},
    },
)
print("Processed table 'orders_summary' registered in Glue.")

tables = [t["Name"] for t in glue.get_tables(DatabaseName="pipeline_db")["TableList"]]
print("Tables in pipeline_db:", tables)

Processed table 'orders_summary' registered in Glue.
Tables in pipeline_db: ['orders', 'orders_summary']


---
## Step 10 — Tear down

In [13]:
# Terminate fleet
ec2.delete_fleets(FleetIds=[fleet_id], TerminateInstances=True)
print("Fleet terminated.")

# Empty and delete S3 buckets
for bucket in [RAW_BUCKET, PROCESSED_BUCKET, RESULTS_BUCKET]:
    objects = helpers.list_all_objects(s3, bucket)
    if objects:
        s3.delete_objects(
            Bucket=bucket,
            Delete={"Objects": [{"Key": o["Key"]} for o in objects]}
        )
    s3.delete_bucket(Bucket=bucket)
    print(f"Deleted: s3://{bucket}")

# Stop the mock
mock.stop()
print("\nMock stopped. Pipeline complete.")

Fleet terminated.
Deleted: s3://pipeline-raw-data
Deleted: s3://pipeline-processed-data
Deleted: s3://pipeline-athena-results

Mock stopped. Pipeline complete.


## Pipeline Architecture Summary

```
┌──────────────────────────────────────────────────────────────────┐
│  INGESTION                                                        │
│  Raw CSV → S3 (pipeline-raw-data/orders/year=Y/month=M/day=D/)  │
└──────────────────┬───────────────────────────────────────────────┘
                   │
                   ▼
┌──────────────────────────────────────────────────────────────────┐
│  CATALOG                                                          │
│  Glue DB: pipeline_db  →  Table: orders (partitioned by y/m/d)  │
└──────────────────┬───────────────────────────────────────────────┘
                   │
                   ▼
┌──────────────────────────────────────────────────────────────────┐
│  QUERY                                                            │
│  Athena workgroup: pipeline-wg  →  Results in athena-results/    │
└──────────────────┬───────────────────────────────────────────────┘
                   │
                   ▼
┌──────────────────────────────────────────────────────────────────┐
│  COMPUTE                                                          │
│  EC2 Fleet: 2 on-demand + 3 spot m5.xlarge/m5.2xlarge workers    │
└──────────────────┬───────────────────────────────────────────────┘
                   │
                   ▼
┌──────────────────────────────────────────────────────────────────┐
│  OUTPUT                                                           │
│  Processed Parquet → S3 (pipeline-processed-data/)               │
│  Glue Table: orders_summary (Parquet SerDe)                      │
└──────────────────────────────────────────────────────────────────┘
```

**Next notebook:** [06_mock_exam.ipynb](06_mock_exam.ipynb)